<a href="https://colab.research.google.com/github/debo-ogunnowo/Prompt-Inference-System/blob/main/prompt_reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers==4.55.2 peft==0.17.0 accelerate==1.10.0 trl==0.21.0 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.1 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.8.0
    Uninstalling safetensors-0.8.0:
      Successfully uninstalled safetensors-0.8.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/FYP/prompt_repsonse_clean.csv')
df.head()

,message,cleaned_response
0,I think I'm mixing metaphors in this paragraph...,"You're close, but mixing metaphors can be tric..."
1,I keep restating my research question in almos...,You can condense the discussion to reduce redu...
2,Can you weave a reference to Table 2 into this...,"Here's a revised paragraph: ""As observed in th..."
3,"This reminder email buries the actual ask, can...",Here's a rewritten version with a clearer call...
4,These items in the sentence don't have paralle...,You're correct that the sentence has non-paral...


In [ ]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

KeyboardInterrupt: 

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

repo_id = 'microsoft/Phi-3-mini-4k-instruct'
model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="cuda:0",
    quantization_config=bnb_config
)

In [ ]:
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj']
)

model = get_peft_model(model, config)
model

In [ ]:
print(model.get_memory_footprint()/1e6)

In [ ]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Trainable parameters:      {train_p/1e6:.2f}M')
print(f'Total parameters:          {tot_p/1e6:.2f}M')
print(f'% of trainable parameters: {100*train_p/tot_p:.2f}%')


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

In [ ]:
train_df.shape[0]

In [ ]:
dataset = load_dataset('csv', data_files='train.csv', split="train")
dataset

In [ ]:
dataset[0]

In [ ]:
# Adapted from trl.extras.dataset_formatting.instructions_formatting_function
# Converts dataset from prompt/completion format (not supported anymore)
# to the conversational format
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}


In [ ]:
dataset = dataset.rename_columns({"cleaned_response": "prompt"})
dataset = dataset.rename_columns({"message": "completion"})
dataset = dataset.map(format_dataset)
dataset = dataset.remove_columns(['prompt', 'completion'])
messages = dataset[0]['messages']
messages

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id
tokenizer.chat_template

In [ ]:
print(tokenizer.apply_chat_template(messages, tokenize=False))

In [ ]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing
    gradient_checkpointing=True,    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=4,
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=8,
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    max_length=1024,
    # Dataset
    # packing a dataset means no padding is needed
    packing=False,
    #packing_strategy='wrapped',

    ## GROUP 3: These are typical training parameters
    num_train_epochs=5,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    optim='paged_adamw_8bit',

    ## GROUP 4: Logging parameters
    logging_steps=10,
    logging_dir='./logs',
    report_to='none',

    # Saving checkpoints
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    output_dir='/content/drive/MyDrive/FYP/prompt-reconstruction-engine_v2',

    # ensures bf16 (the new default) is only used when it is actually available
    bf16=torch.cuda.is_bf16_supported(including_emulation=False)
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=config,
    processing_class=tokenizer,
    args=sft_config,
)

In [ ]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [ ]:
batch['input_ids'][0], batch['labels'][0]

In [ ]:
trainer.train()

In [ ]:
def gen_prompt(tokenizer, sentence):
  converted_sample = [{"role": "user", "content": sentence}]
  prompt = tokenizer.apply_chat_template(
      converted_sample, tokenize=False, add_generation_prompt=True
  )
  return prompt

In [ ]:
sentence = "Hunger – A Global Challenge Hunger is the physiological and psychological need for food that occurs when the body’s energy demands exceed its intake. While a normal human appetite is a natural cue for feeding, chronic hunger—known as under‑nutrition—affects millions worldwide and can persist for years. It is not merely a lack of calories; it often reflects inadequate access to diverse, nutritious foods, leading to deficiencies in protein, vitamins, and minerals. The roots of hunger are complex. Poverty, conflict, climate change, and unequal food distribution all play a part. In many regions, seasonal droughts or flooding destroy crops, while in others, market forces keep staples unaffordable for the poorest. Food waste adds to the paradox; an estimated one‑third of all food produced is never eaten, yet countless families starve. Consequences extend beyond physical health. Children who grow up in food‑scarce environments experience stunted growth, weakened immune systems, and impaired learning, trapping generations in a cycle of poverty. Adults suffer from reduced productivity and increased vulnerability to disease. Socially, chronic hunger can spark migration, unrest, and economic instability. Addressing hunger demands a multifaceted response: investing in resilient agriculture, improving supply chains, ensuring fair trade, and promoting nutrition‑education programs. Community gardens, micro‑loans for farmers, and local food banks can bridge gaps in the short term, while long‑term policy reforms must tackle the structural drivers of inequality. By combining immediate relief with systemic change, we can move toward a world where every person has reliable access to sufficient, nutritious food."

In [ ]:
prompt = gen_prompt(tokenizer, sentence)
print(prompt)

In [ ]:
from contextlib import nullcontext

def generate(model, tokenizer, prompt, max_new_tokens=1024, skip_special_tokens=False):
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    # if it was trained using mixed precision, uses autocast context
    ctx = torch.autocast(device_type=model.device.type, dtype=model.dtype) \
          if model.dtype in [torch.float16, torch.bfloat16] else nullcontext()
    with ctx:
        gen_output = model.generate(**tokenized_input,
                                    eos_token_id=tokenizer.eos_token_id,
                                    max_new_tokens=max_new_tokens)

    output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)
    return output[0]


In [ ]:
def get_assistant_response(full_output):
    # Find the start and end markers for assistant's response
    start_marker = "<|assistant|>"
    end_marker = "<|end|>"

    # Find positions
    start_idx = full_output.find(start_marker)
    if start_idx == -1:
        return ""

    # Move past the start marker
    start_idx += len(start_marker)

    # Find the end marker after the start
    end_idx = full_output.find(end_marker, start_idx)
    if end_idx == -1:
        return full_output[start_idx:]

    return full_output[start_idx:end_idx].strip()

full_output = generate(model, tokenizer, prompt)
assistant_response = get_assistant_response(full_output)
print(assistant_response)

In [ ]:
# Save model
model.save_pretrained('/content/drive/MyDrive/FYP/prompt_reconstruction_model_v2')
tokenizer.save_pretrained('/content/drive/MyDrive/FYP/prompt_reconstruction_model_v2')
print("Model saved.")

In [ ]:
test_df = pd.read_csv('test.csv')
test_df.head()


In [ ]:
# Final model eval
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

repo_id = 'microsoft/Phi-3-mini-4k-instruct'

adapter_path = '/content/drive/MyDrive/FYP/prompt_reconstruction_model'

tokenizer = AutoTokenizer.from_pretrained(repo_id)
base_model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    torch_dtype=torch.float16,
    device_map='auto'
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=False
)
model = model.merge_and_unload()
model.eval()

def reconstruct_prompt(response, max_new_tokens=100):
  messages = [{"role": "user", "content": response}]
  input_text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )
  inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
  with torch.no_grad():
    outputs = model.generate(
        **inputs,
          max_new_tokens=max_new_tokens,
          do_sample=False,
          temperature=1.0,
          pad_token_id=tokenizer.eos_token_id
    )
  # Decode only newly generated tokens
  generated = outputs[0][inputs['input_ids'].shape[1]:]
  return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
reconstructed = []
for i, row in test_df.iterrows():
  inferred = reconstruct_prompt(row['cleaned_response'])
  reconstructed.append(inferred)
  if (i + 1) % 10 == 0:
    print(f"Processed {i + 1}/{len(test_df)}")

test_df['reconstructed_prompt'] = reconstructed

test_df.to_csv('prompt_reconstruction_results.csv', index=False)
print('Prompt Reconstruction Complete. Results Saved')


In [ ]:
df1 = pd.read_csv('prompt_reconstruction_results.csv')


df1.head()

In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import numpy as np

print('Sample Pairs: ')
for i in range(3):
  print(f"\nOriginal: {df1.iloc[i]['message']}")
  print(f"Reconstructed: {df1.iloc[i]['reconstructed_prompt']}")

In [ ]:
# Eval 1: Semantic similarity

print("\n Evaluating semantic similarity...")
sem_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

original_embeddings = sem_model.encode(
    df1['message'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

reconstructed_embeddings = sem_model.encode(
    df1['reconstructed_prompt'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

similarity_scores = util.cos_sim(
    original_embeddings,
    reconstructed_embeddings
).diagonal().cpu().numpy()

df1['semantic similarity'] = similarity_scores

print(f"Mean semantic similarity:       {similarity_scores.mean():.4f}")
print(f"Median semantic similarity:     {np.median(similarity_scores):.4f}")
print(f"Proportion above 0.7:           {(similarity_scores > 0.7).mean():.4f}")
print(f"Proportion above 0.5:           {(similarity_scores > 0.5).mean():.4f}")

In [ ]:
df1.head()

In [ ]:
# Eval 2: Category Agreement
import torch

print("\n Evaluating category agreement...")
label_map = {
    'LABEL_0': 'Research&Inquiry',
    'LABEL_1': 'Content Generation',
    'LABEL_2': 'Text Refinement'
}

classifier = pipeline(
    'text-classification',
    model='/content/drive/MyDrive/FYP/prompt_classifier',
    tokenizer='/content/drive/MyDrive/FYP/prompt_classifier',
    device=0 if torch.cuda.is_available() else -1
)

def get_category(text):
  try:
    result = classifier(
      str(text),
      truncation=True,
      max_length=128
    )[0]
    return label_map[result['label']]
  except Exception as e:
    return 'Unknown'

df1['original_category'] = df1['message'].apply(get_category)
df1['reconstructed_category'] = df1['reconstructed_prompt'].apply(get_category)
df1['category_match'] = (
    df1['original_category'] == df1['reconstructed_category']
)

agreement_rate = df1['category_match'].mean()
print(f"Overall category agreement rate: {agreement_rate:.4f}, {agreement_rate:.2%}")
print("\nAgreement rate by original category:")
print(df1.groupby('original_category')['category_match'].mean())

df1.to_csv('/content/drive/MyDrive/FYP/final_reconstruction_evaluation_results.csv', index=False)
print("\nFull evaluation results saved.")

